[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C16_Generative_Models_Course/04_diffusion/04_diffusion.ipynb)

# 04 · 扩散模型 DDPM（用 numpy 从零）

目标：从零实现 **DDPM**——前向加噪闭式、预测噪声的网络 ε_θ、简化 MSE 损失、反向采样链，在玩具双峰分布上训练并 **从纯噪声采样生成**；理解噪声调度。

路线：噪声调度 + 前向闭式(**对拍逐步加噪**) → ε_θ 网络 + 简化损失(**梯度检验**) → 训练 → **反向采样收敛** → 噪声调度/SNR → ✏️ 练习（前向 q、损失、反向步、调度）→ 📖 答案 → 🧪 真实(cosine 调度)胶囊。

> 心智模型：**把生成拆成许多步小去噪。前向是固定公式(无参数)，唯一要学的是「预测噪声」的网络。稳如监督回归、却能锐利采样**。

## 1 · 噪声调度 + 前向闭式（对拍逐步加噪）

线性调度 `β_t`，累积 `ᾱ_t=∏α_s`。前向闭式：`x_t = √ᾱ_t·x_0 + √(1-ᾱ_t)·ε`。

**验证闭式正确**：直接用闭式算 `x_t`，与老老实实逐步加噪 `t` 步，给出**统计一致**的分布。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

T = 50
betas = np.linspace(1e-4, 0.06, T)      # 线性噪声调度（玩具：T 小所以 βmax 调大些，让末步接近纯噪声）
alphas = 1 - betas
alpha_bars = np.cumprod(alphas)         # ᾱ_t

def q_sample(x0, t, eps):
    '''前向闭式：一步加噪到时刻 t。x0:(N,1), t:int, eps:(N,1)'''
    ab = alpha_bars[t]
    return np.sqrt(ab) * x0 + np.sqrt(1 - ab) * eps

def q_stepwise(x0, t, g):
    '''逐步加噪 t+1 步（t=0 表示走 1 步），用于对拍闭式。'''
    x = x0.copy()
    for s in range(t + 1):
        x = np.sqrt(1 - betas[s]) * x + np.sqrt(betas[s]) * g.standard_normal(x.shape)
    return x

# 调度基本性质
assert np.all(np.diff(alpha_bars) < 0), 'ᾱ_t 应单调下降'
assert alpha_bars[0] > 0.99 and alpha_bars[-1] < 0.3, '首步≈全信号、末步信号大减(接近纯噪声)'
# 对拍：闭式 vs 逐步（比较同一 t 处的分布统计）
x0 = 2.0 * np.ones((20000, 1))                 # 固定 x0 看加噪分布
t = 30
x_closed = q_sample(x0, t, rng.standard_normal((20000, 1)))
x_step = q_stepwise(x0, t, np.random.default_rng(1))
print('t=%d 闭式:  均值=%.3f 方差=%.3f' % (t, x_closed.mean(), x_closed.var()))
print('t=%d 逐步:  均值=%.3f 方差=%.3f' % (t, x_step.mean(), x_step.var()))
# 理论：均值=√ᾱ_t·x0, 方差=1-ᾱ_t
assert abs(x_closed.mean() - np.sqrt(alpha_bars[t])*2.0) < 0.05
assert abs(x_closed.var() - (1-alpha_bars[t])) < 0.05
assert abs(x_closed.mean() - x_step.mean()) < 0.05, '闭式与逐步均值一致'
assert abs(x_closed.var() - x_step.var()) < 0.05, '闭式与逐步方差一致'
print('✅ 前向闭式 == 逐步加噪（统计一致）；这就是 DDPM 高效训练的命脉')

## 2 · 预测噪声的网络 ε_θ + 简化损失（梯度检验）

网络输入 `[x_t, t]`（把时刻 t 归一化拼上去），输出预测的噪声。
简化损失：`L = E‖ε - ε_θ(x_t, t)‖²`——一个朴素的回归。手写反向，数值梯度检验守住。

In [ ]:
def init_eps_net(H=64, seed=0, s=0.5):
    g = np.random.default_rng(seed)
    return dict(W1=s*g.standard_normal((2,H)), b1=np.zeros(H),
                W2=s*g.standard_normal((H,H)), b2=np.zeros(H),
                W3=s*g.standard_normal((H,1)), b3=np.zeros(1))

def eps_forward(xt, tnorm, P):
    '''xt:(N,1), tnorm:(N,1) in [0,1]. 返回预测噪声 (N,1) 与缓存。'''
    inp = np.concatenate([xt, tnorm], axis=1)
    h1 = np.tanh(inp @ P['W1'] + P['b1'])
    h2 = np.tanh(h1 @ P['W2'] + P['b2'])
    out = h2 @ P['W3'] + P['b3']
    return out, (inp, h1, h2)

def eps_loss_and_grads(xt, tnorm, eps, P):
    N = xt.shape[0]
    out, (inp, h1, h2) = eps_forward(xt, tnorm, P)
    loss = np.mean(np.sum((out - eps)**2, axis=1))
    dout = (2.0 / N) * (out - eps)
    g = {}
    g['W3'] = h2.T @ dout; g['b3'] = dout.sum(0)
    dh2 = (dout @ P['W3'].T) * (1 - h2**2)
    g['W2'] = h1.T @ dh2; g['b2'] = dh2.sum(0)
    dh1 = (dh2 @ P['W2'].T) * (1 - h1**2)
    g['W1'] = inp.T @ dh1; g['b1'] = dh1.sum(0)
    return loss, g

P = init_eps_net()
xt = rng.standard_normal((32, 1)); tn = rng.random((32, 1)); ep = rng.standard_normal((32, 1))
loss, grads = eps_loss_and_grads(xt, tn, ep, P)
# 数值梯度检验
maxerr = 0.0
for key in ['W1', 'W2', 'W3']:
    W = P[key]; gn = np.zeros_like(W); it = np.nditer(W, flags=['multi_index'])
    while not it.finished:
        i = it.multi_index; o = W[i]
        W[i]=o+1e-6; fp=eps_loss_and_grads(xt,tn,ep,P)[0]
        W[i]=o-1e-6; fm=eps_loss_and_grads(xt,tn,ep,P)[0]; W[i]=o
        gn[i]=(fp-fm)/2e-6; it.iternext()
    maxerr = max(maxerr, np.max(np.abs(gn - grads[key])))
print('梯度检验 max|err| = %.2e' % maxerr)
assert maxerr < 1e-5, 'ε_θ 反向应与数值梯度一致'
print('✅ 预测噪声网络 + 简化 MSE 损失就位，反向写对了')

## 3 · 训练 ε_θ：随机抽 t、加噪、预测噪声

训练循环朴素得像监督学习：每步抽一批 `x_0`、给每个样本随机抽 `t` 和 `ε`、闭式合成 `x_t`、预测 `ε`、MSE 下降。

**注意稳定性**：这是个普通回归，损失会平滑下降——没有 GAN 的振荡、没有 VAE 的 KL 平衡。

In [ ]:
def make_bimodal(n, g, sep=2.0, noise=0.3):
    comp = g.integers(0, 2, n)
    return (np.where(comp==0, -sep, sep) + noise*g.standard_normal(n)).reshape(-1, 1)

def train_ddpm(epochs=3000, lr=0.01, H=64, bs=256, seed=0):
    P = init_eps_net(H=H, seed=seed)
    g = np.random.default_rng(7); hist = []
    for ep in range(epochs):
        x0 = make_bimodal(bs, g)
        t = g.integers(0, T, bs)                     # 每样本随机时刻
        eps = g.standard_normal((bs, 1))
        ab = alpha_bars[t].reshape(-1, 1)
        xt = np.sqrt(ab) * x0 + np.sqrt(1 - ab) * eps  # 前向闭式
        tn = (t / T).reshape(-1, 1)
        loss, grads = eps_loss_and_grads(xt, tn, eps, P)
        for k in P: P[k] -= lr * grads[k]
        hist.append(loss)
    return P, hist

P_t, hist = train_ddpm(epochs=4000, lr=0.01)
early = np.mean(hist[:200]); late = np.mean(hist[-200:])
print('简化损失(平滑): %.3f -> %.3f' % (early, late))
assert late < early, '训练应降低预测噪声的 MSE'
# 平滑后应大体单调（监督回归的稳定性）
win = [np.mean(hist[i:i+200]) for i in range(0, len(hist)-200, 200)]
assert win[-1] < win[0], '损失应稳定下降（无 GAN 式振荡）'
print('✅ ε_θ 训练完成，损失平滑下降 —— 扩散训练稳如监督学习')

## 4 · 反向采样：从纯噪声生成数据

从 `x_T~N(0,I)` 出发，对 `t=T-1...0` 逐步去噪：
`x_{t-1} = (x_t - (1-α_t)/√(1-ᾱ_t)·ε_θ)/√α_t + σ_t·z`（最后一步不加噪）。

采样后看生成分布是否收敛到目标双峰——这是扩散「生成」的本体。

In [ ]:
def ddpm_sample(P, n, seed=0):
    g = np.random.default_rng(seed)
    x = g.standard_normal((n, 1))                    # x_T ~ N(0,I)
    for t in range(T - 1, -1, -1):
        tn = np.full((n, 1), t / T)
        eps_pred, _ = eps_forward(x, tn, P)
        a = alphas[t]; ab = alpha_bars[t]
        mean = (1.0/np.sqrt(a)) * (x - (1 - a)/np.sqrt(1 - ab) * eps_pred)
        if t > 0:
            x = mean + np.sqrt(betas[t]) * g.standard_normal((n, 1))   # 加随机性
        else:
            x = mean                                # 最后一步不加噪
    return x

def hist_l1(a, b, bins=50, rr=(-5, 5)):
    ha, e = np.histogram(a, bins=bins, range=rr, density=True)
    hb, _ = np.histogram(b, bins=bins, range=rr, density=True)
    return float(np.sum(np.abs(ha - hb)) * (e[1]-e[0]))

real = make_bimodal(3000, np.random.default_rng(5)).ravel()
fake = ddpm_sample(P_t, 3000, seed=99).ravel()
d = hist_l1(real, fake)
print('真实  均值=%.2f std=%.2f' % (real.mean(), real.std()))
print('生成  均值=%.2f std=%.2f' % (fake.mean(), fake.std()))
print('分布距离 L1 = %.3f （扩散通常 < GAN！）' % d)
print('生成谷底 |x|<1 比例 = %.3f (双峰)' % np.mean(np.abs(fake)<1))
assert d < 0.35, '生成分布应收敛到目标双峰'
assert abs(fake.std() - real.std()) < 0.4
assert np.mean(np.abs(fake)<1) < 0.15, '应学出清晰双峰'
assert 0.3 < np.mean(fake < 0) < 0.7, '应均衡覆盖两峰（扩散不易坍塌）'
print('✅ DDPM 从纯噪声采样 -> 双峰分布，距离比 GAN 更小、覆盖更全、过程更稳！')

## 5 · 噪声调度与信噪比 SNR

调度 `{β_t}` 决定信号如何随 `t` 被噪声淹没。统一语言是**信噪比** `SNR(t)=ᾱ_t/(1-ᾱ_t)`，随 `t` 单调下降。
看看信号系数 `√ᾱ_t` 与 SNR 如何随时间变化。

In [ ]:
snr = alpha_bars / (1 - alpha_bars)
signal_coef = np.sqrt(alpha_bars)
noise_coef = np.sqrt(1 - alpha_bars)
print(f"{'t':>4} {'√ᾱ_t(信号)':>12} {'√(1-ᾱ_t)(噪声)':>16} {'SNR':>10}")
for t in [0, 10, 25, 40, T-1]:
    print(f'{t:>4} {signal_coef[t]:>12.3f} {noise_coef[t]:>16.3f} {snr[t]:>10.3f}')
# 性质：信号系数单调降、噪声系数单调升、SNR 单调降；首尾趋近
assert np.all(np.diff(signal_coef) < 0), '信号系数应单调下降'
assert np.all(np.diff(snr) < 0), 'SNR 应单调下降'
assert signal_coef[0] > 0.99 and signal_coef[-1] < 0.5
# 信号方差 + 噪声方差 = 1（守恒）
assert np.allclose(signal_coef**2 + noise_coef**2, 1.0), '信号方差+噪声方差=1'
print('✅ 调度 = 设计 SNR 如何随 t 下降；信号渐隐、噪声渐显、方差守恒')

## 6 · 预测噪声 ⇔ 预测 x_0（一体两面）

从前向闭式 `x_t=√ᾱ_t·x_0+√(1-ᾱ_t)·ε` 反解：知道了 `ε`（或 `x_0`）就知道另一个。
所以「预测噪声 ε」和「预测原图 x_0」是**等价**的参数化。验证两者可互相换算。

In [ ]:
def predict_x0_from_eps(xt, eps, t):
    '''由预测噪声反解 x_0：x_0 = (x_t - √(1-ᾱ_t)·ε)/√ᾱ_t'''
    ab = alpha_bars[t]
    return (xt - np.sqrt(1 - ab) * eps) / np.sqrt(ab)

def predict_eps_from_x0(xt, x0, t):
    '''由预测 x_0 反解噪声：ε = (x_t - √ᾱ_t·x_0)/√(1-ᾱ_t)'''
    ab = alpha_bars[t]
    return (xt - np.sqrt(ab) * x0) / np.sqrt(1 - ab)

# 造一个 (x0, eps, xt) 三元组，验证互相反解一致
x0_true = make_bimodal(100, np.random.default_rng(3))
eps_true = rng.standard_normal((100, 1))
t = 20
xt = q_sample(x0_true, t, eps_true)
x0_rec = predict_x0_from_eps(xt, eps_true, t)
eps_rec = predict_eps_from_x0(xt, x0_true, t)
assert np.allclose(x0_rec, x0_true, atol=1e-8), '由 ε 应能精确反解 x_0'
assert np.allclose(eps_rec, eps_true, atol=1e-8), '由 x_0 应能精确反解 ε'
print('✅ 预测噪声 ⇔ 预测 x_0：一体两面，可精确互换（DDPM 用 ε，有些工作用 x_0/v）')

---
## ✏️ 练习 1：前向加噪 q(x_t|x_0)

实现 `forward_diffuse(x0, t, eps, alpha_bars)`：用闭式 `√ᾱ_t·x0 + √(1-ᾱ_t)·ε` 一步加噪到时刻 `t`。

In [ ]:
def forward_diffuse(x0, t, eps, alpha_bars):
    # TODO: ab = alpha_bars[t]; 返回 √ab·x0 + √(1-ab)·eps
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
x0 = np.ones((10000, 1)) * 3.0
# t=0: ᾱ≈α_0≈1 -> 几乎无噪，x_t≈x_0
xt0 = forward_diffuse(x0, 0, rng.standard_normal((10000,1)), alpha_bars)
assert abs(xt0.mean() - 3.0*np.sqrt(alpha_bars[0])) < 0.05
# t=T-1: 信号大减、方差接近 1-ᾱ_{T-1}
xtT = forward_diffuse(x0, T-1, rng.standard_normal((10000,1)), alpha_bars)
assert abs(xtT.var() - (1-alpha_bars[T-1])) < 0.06
assert xtT.std() > xt0.std(), '越往后噪声越大'
print('✅ 练习 1 通过：前向闭式加噪正确')

## ✏️ 练习 2：噪声预测损失

实现 `ddpm_loss(eps_true, eps_pred)`：简化目标 `mean ‖ε - ε_θ‖²`（每样本平方和、再对样本平均）。

In [ ]:
def ddpm_loss(eps_true, eps_pred):
    # TODO: mean_n sum_d (eps_true - eps_pred)^2
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
e = rng.standard_normal((50, 1))
assert abs(ddpm_loss(e, e)) < 1e-12, '完美预测损失为 0'
# 预测全 0 时，损失 = mean ‖ε‖² ≈ 维度(=1)（因为 ε~N(0,1)）
big = rng.standard_normal((100000, 1))
assert abs(ddpm_loss(big, np.zeros_like(big)) - 1.0) < 0.05, '预测0时损失≈E‖ε‖²=1'
print('✅ 练习 2 通过：噪声预测 MSE 正确（预测0时≈1）')

## ✏️ 练习 3：单步反向采样

实现 `reverse_step(xt, eps_pred, t, alphas, alpha_bars, betas, z)`：执行一步去噪 `(xt - (1-α_t)/√(1-ᾱ_t)·ε_pred)/√α_t + (√β_t·z if t>0 else 0)`。返回 `x_{t-1}`。

In [ ]:
def reverse_step(xt, eps_pred, t, alphas, alpha_bars, betas, z):
    # TODO: a=alphas[t]; ab=alpha_bars[t]
    #       mean = (xt - (1-a)/√(1-ab)·eps_pred)/√a
    #       return mean + (√betas[t]·z if t>0 else 0)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
xt = rng.standard_normal((20, 1)); ep = rng.standard_normal((20, 1)); z = rng.standard_normal((20, 1))
# t=0 不加噪：结果应是确定的均值
x_prev0 = reverse_step(xt, ep, 0, alphas, alpha_bars, betas, z)
a, ab = alphas[0], alpha_bars[0]
expected = (xt - (1-a)/np.sqrt(1-ab)*ep)/np.sqrt(a)
assert np.allclose(x_prev0, expected), 't=0 应为纯均值(无噪)'
# t>0 加噪：与不加噪的均值相差 √β_t·z
x_prev5 = reverse_step(xt, ep, 5, alphas, alpha_bars, betas, z)
a5, ab5 = alphas[5], alpha_bars[5]
mean5 = (xt - (1-a5)/np.sqrt(1-ab5)*ep)/np.sqrt(a5)
assert np.allclose(x_prev5 - mean5, np.sqrt(betas[5])*z), 't>0 应加 √β_t·z'
print('✅ 练习 3 通过：单步反向采样正确（t=0 不加噪、t>0 加噪）')

## ✏️ 练习 4：构造噪声调度

实现 `make_schedule(T, beta_start, beta_end)`：返回线性 `betas` 及由它算出的 `alphas`、`alpha_bars`。验证 `ᾱ_t` 单调下降、且 `alphas=1-betas`、`alpha_bars=cumprod(alphas)`。

In [ ]:
def make_schedule(T, beta_start=1e-4, beta_end=0.02):
    # TODO: betas=linspace(...); alphas=1-betas; alpha_bars=cumprod(alphas); 返回三者
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
b, a, ab = make_schedule(100)
assert len(b) == 100 and np.allclose(a, 1 - b)
assert np.allclose(ab, np.cumprod(a))
assert np.all(np.diff(ab) < 0), 'ᾱ_t 单调下降'
assert ab[0] > 0.999 and ab[-1] < ab[0], '首步几乎无噪、之后累积加噪'
print('✅ 练习 4 通过：噪声调度构造正确')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def forward_diffuse(x0, t, eps, alpha_bars):
    ab = alpha_bars[t]
    return np.sqrt(ab) * x0 + np.sqrt(1 - ab) * eps

In [ ]:
# 练习 2 参考答案
def ddpm_loss(eps_true, eps_pred):
    return np.mean(np.sum((eps_true - eps_pred)**2, axis=1))

In [ ]:
# 练习 3 参考答案
def reverse_step(xt, eps_pred, t, alphas, alpha_bars, betas, z):
    a = alphas[t]; ab = alpha_bars[t]
    mean = (xt - (1 - a)/np.sqrt(1 - ab) * eps_pred) / np.sqrt(a)
    if t > 0:
        return mean + np.sqrt(betas[t]) * z
    return mean

In [ ]:
# 练习 4 参考答案
def make_schedule(T, beta_start=1e-4, beta_end=0.02):
    betas = np.linspace(beta_start, beta_end, T)
    alphas = 1 - betas
    alpha_bars = np.cumprod(alphas)
    return betas, alphas, alpha_bars

---
## 🧪 真实调度胶囊：cosine 调度（Nichol & Dhariwal 2021）

DDPM 原版用线性调度，但 Nichol & Dhariwal 2021 发现 **cosine 调度**（让 ᾱ_t 按余弦下降）在标准的大 T 设置下加噪更均匀、并能**更彻底地把信号噪声化**（末段 ᾱ_t 更接近 0）。这里构造它并对比。**纯本地计算**。

In [ ]:
def cosine_alpha_bars(T, s=0.008):
    '''Nichol & Dhariwal 2021 的 cosine 调度：ᾱ_t = f(t)/f(0), f(t)=cos²((t/T+s)/(1+s)·π/2)'''
    ts = np.arange(T + 1)
    f = np.cos((ts / T + s) / (1 + s) * np.pi / 2) ** 2
    ab = f / f[0]
    return ab[1:]                       # 取 t=1..T

ab_cos = cosine_alpha_bars(T)
ab_lin = alpha_bars
print(f"{'t':>4} {'线性 ᾱ_t':>12} {'cosine ᾱ_t':>12}")
for t in [0, 12, 25, 37, T-1]:
    print(f'{t:>4} {ab_lin[t]:>12.3f} {ab_cos[t]:>12.3f}')
print('cosine 末步 ᾱ=%.4f（≈0，信号被彻底噪声化）vs 线性末步 ᾱ=%.4f' % (ab_cos[-1], ab_lin[-1]))
print('✅ cosine 调度就绪（末段把信号更彻底地噪声化 -> x_T 更接近纯 N(0,I)）')

**🧪 胶囊练习**：实现 `is_valid_schedule(alpha_bars)`：验证一个调度合法——`ᾱ_t` 都在 (0,1]、单调不增、且首值接近 1。返回 bool。

In [ ]:
def is_valid_schedule(alpha_bars):
    # TODO: 全在(0,1]; 单调不增(diff<=eps); 首值>0.9; 返回 bool
    raise NotImplementedError

In [ ]:
# 自测
assert is_valid_schedule(ab_lin), '线性调度应合法'
assert is_valid_schedule(ab_cos), 'cosine 调度应合法'
# cosine 末段把信号更彻底地噪声化（终值更接近 0 -> x_T 更接近纯噪声）
assert ab_cos[-1] < ab_lin[-1], 'cosine 末步 ᾱ 更小(信号更彻底被噪声化)'
# 非法调度被拒
assert not is_valid_schedule(np.array([0.5, 0.6, 0.7])), '上升的调度非法'
print('✅ 胶囊练习通过：能校验调度合法性，并看出 cosine vs 线性的差异')

In [ ]:
# 📖 胶囊参考答案
def is_valid_schedule(alpha_bars):
    ab = np.asarray(alpha_bars)
    in_range = np.all((ab > 0) & (ab <= 1.0 + 1e-9))
    monotone = np.all(np.diff(ab) <= 1e-9)
    starts_high = ab[0] > 0.9
    return bool(in_range and monotone and starts_high)

### 小结
- 扩散 = 前向**固定**逐步加噪（有一步到位的闭式 `√ᾱ_t·x0+√(1-ᾱ_t)·ε`）+ **可学**逐步去噪。
- 唯一要训练的是**预测噪声**的网络 ε_θ，目标是朴素的 MSE——**稳如监督回归**。
- 反向采样从纯噪声逐步去噪；生成分布距离比 GAN 更小、覆盖更全、**不易坍塌**。
- **噪声调度**（线性/cosine）= 设计 SNR 如何随 t 下降，对质量影响大。
- 预测噪声 ⇔ 预测 x_0 ⇔ 估计 score：一体多面，并把扩散与 VAE(变分界)、去噪 AE(score) **统一**。

下一站：**模块 05 · 流匹配** —— 取扩散的连续极限，把去噪看成沿速度场流动，统一扩散与流、采样可极少步。